In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import gamma, norm


CELL 1: LOAD RECOVERED SILVER DATA & FILTER LAND-ONLY GRIDS

In [4]:
# 1. Đường dẫn tới file Silver Parquet đã khôi phục ĐẦY ĐỦ 312 tháng
SILVER_PARQUET = Path(
    "/home/trang/Drought_prediction/data/silver/era5_land/era5_land_mekong_delta_2000_2025.parquet"
)
GOLD_DIR = Path("/home/trang/Drought_prediction/data/gold/features")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# 2. Đọc Silver Parquet
print("► Đang nạp dữ liệu Silver Layer chuẩn...")
silver_df = pd.read_parquet(SILVER_PARQUET)

# 3. Lọc sạch NaN (loại bỏ ô biển, giữ đúng 326 ô đất liền ĐBSCL)
gold_base_df = silver_df.dropna(subset=["temp_mean_celsius"]).copy()
gold_base_df = gold_base_df.sort_values(
    by=["latitude", "longitude", "valid_time"]
).reset_index(drop=True)

# 4. Kiểm tra đối soát số lượng tháng và ô lưới
total_land_grids = (
    gold_base_df[["latitude", "longitude"]].drop_duplicates().shape[0]
)
total_months = gold_base_df["valid_time"].nunique()

print("\n==================================================")
print(" NGHIỆM THU ĐẦU VÀO TẦNG GOLD (DỮ LIỆU ĐÃ RECOVER)")
print("==================================================")
print(f"• Số ô lưới đất liền ĐBSCL : {total_land_grids} ô (Chuẩn 326)")
print(f"• Chuỗi thời gian           : {total_months} tháng (Đủ 312 tháng từ 2000-2025)")
print(f"• Tổng số bản ghi Gold base : {len(gold_base_df):,} bản ghi (326 x 312)")

► Đang nạp dữ liệu Silver Layer chuẩn...

 NGHIỆM THU ĐẦU VÀO TẦNG GOLD (DỮ LIỆU ĐÃ RECOVER)
• Số ô lưới đất liền ĐBSCL : 326 ô (Chuẩn 326)
• Chuỗi thời gian           : 312 tháng (Đủ 312 tháng từ 2000-2025)
• Tổng số bản ghi Gold base : 101,712 bản ghi (326 x 312)


Step 02 - Calculate SPI & SPEI Drought Indices

In [5]:
def calculate_spi(series, scale=3):
    """Tính chỉ số SPI (Standardized Precipitation Index) dựa trên phân bố Gamma"""
    rolling_precip = series.rolling(window=scale, min_periods=scale).sum()

    # Tính SPI cho từng ô lưới dựa trên chuẩn hóa Z-score / Gamma
    def fit_gamma_spi(x):
        x_valid = x.dropna()
        if len(x_valid) < 30:  # Cần đủ mẫu để fit phân bố
            return pd.Series(np.nan, index=x.index)

        # Xử lý giá trị 0 (Zero precipitation)
        zeros = x_valid == 0
        q = zeros.mean()

        if q < 1 and not zeros.all():
            params = gamma.fit(x_valid[~zeros])
            cdf = gamma.cdf(x_valid, *params)
            cdf = q + (1 - q) * cdf
            # Ép biên tránh tràn vô cực
            cdf = np.clip(cdf, 0.00001, 0.99999)
            spi = norm.ppf(cdf)
        else:
            spi = np.full_like(x_valid, np.nan)

        res = pd.Series(np.nan, index=x.index)
        res.loc[x_valid.index] = spi
        return res

    return rolling_precip


# 1. Tính cân bằng nước D = Precip - PET cho SPEI
gold_base_df["water_balance_D"] = (
    gold_base_df["precip_monthly_mm"] - gold_base_df["pet_monthly_mm"]
)

# 2. Tính lũy kế lượng mưa và cân bằng nước theo các chu kỳ 1, 3, 6 tháng cho từng ô lưới
print("► Đang tính toán lũy kế và chỉ số SPI/SPEI (1, 3, 6 tháng)...")

for k in [1, 3, 6]:
    # Lũy kế theo nhóm tọa độ địa lý
    gold_base_df[f"precip_roll_{k}m"] = gold_base_df.groupby(
        ["latitude", "longitude"]
    )["precip_monthly_mm"].transform(lambda x: x.rolling(k, min_periods=k).sum())

    gold_base_df[f"water_balance_roll_{k}m"] = gold_base_df.groupby(
        ["latitude", "longitude"]
    )["water_balance_D"].transform(lambda x: x.rolling(k, min_periods=k).sum())

    # Standardized Anomaly Index đơn giản hóa cho chuỗi khí hậu local
    gold_base_df[f"SPI_{k}m"] = gold_base_df.groupby(["latitude", "longitude"])[
        f"precip_roll_{k}m"
    ].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    gold_base_df[f"SPEI_{k}m"] = gold_base_df.groupby(
        ["latitude", "longitude"]
    )[f"water_balance_roll_{k}m"].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-6)
    )

print("Đã hoàn thành tính toán các chỉ số SPI-1, SPI-3, SPI-6, SPEI-1, SPEI-3, SPEI-6!")
display(
    gold_base_df[
        [
            "valid_time",
            "latitude",
            "longitude",
            "precip_monthly_mm",
            "SPI_3m",
            "SPEI_3m",
        ]
    ].tail(5)
)

► Đang tính toán lũy kế và chỉ số SPI/SPEI (1, 3, 6 tháng)...
Đã hoàn thành tính toán các chỉ số SPI-1, SPI-3, SPI-6, SPEI-1, SPEI-3, SPEI-6!


,valid_time,latitude,longitude,precip_monthly_mm,SPI_3m,SPEI_3m
101707,2025-08-01,11.0,106.3,279.741167,0.977606,0.902159
101708,2025-09-01,11.0,106.3,301.975107,1.048537,0.977255
101709,2025-10-01,11.0,106.3,246.126713,1.168001,1.169254
101710,2025-11-01,11.0,106.3,170.775104,0.809504,0.952728
101711,2025-12-01,11.0,106.3,56.055782,0.000432,0.225092


03 - CREATE LAG FEATURES & PUBLISH GOLD MART

In [6]:

print("► Đang khởi tạo các tính năng trễ (Lag Features t-1, t-2, t-3)...")
# Các biến cần tạo trễ cho mô hình học chuỗi thời gian
feature_cols = [
    "temp_mean_celsius",
    "precip_monthly_mm",
    "pet_monthly_mm",
    "SPI_3m",
    "SPEI_3m",
]

for col in feature_cols:
    for lag in [1, 2, 3]:
        gold_base_df[f"{col}_lag_{lag}"] = gold_base_df.groupby(
            ["latitude", "longitude"]
        )[col].shift(lag)

# Loại bỏ các dòng đầu bị thiếu do trễ rolling/lag (khoảng 6 tháng đầu từ năm 2000)
gold_final_df = gold_base_df.dropna(subset=["SPI_3m_lag_3"]).copy()

# Đường dẫn file Gold hoàn chỉnh
GOLD_PARQUET_PATH = GOLD_DIR / "gold_drought_features.parquet"
gold_final_df.to_parquet(GOLD_PARQUET_PATH, index=False)

print("\n==================================================")
print(" HOÀN TẤT XUẤT BẢN TẦNG GOLD (READY FOR XGBOOST!)")
print("==================================================")
print(f"► File Gold Mart đã lưu tại: {GOLD_PARQUET_PATH}")
print(f"• Số lượng dòng dữ liệu hoàn chỉnh : {len(gold_final_df):,} bản ghi")
print(f"• Tổng số cột/tính năng (Features)  : {gold_final_df.shape[1]} cột")

► Đang khởi tạo các tính năng trễ (Lag Features t-1, t-2, t-3)...

 HOÀN TẤT XUẤT BẢN TẦNG GOLD (READY FOR XGBOOST!)
► File Gold Mart đã lưu tại: /home/trang/Drought_prediction/data/gold/features/gold_drought_features.parquet
• Số lượng dòng dữ liệu hoàn chỉnh : 100,082 bản ghi
• Tổng số cột/tính năng (Features)  : 46 cột


04. LỌC VÀ CHỌN CÁC TÍNH NĂNG CHUẨN CHO BÀI TOÁN MACHINE LEARNING / DEEP LEARNING

In [7]:

GOLD_DIR = Path("/home/trang/Drought_prediction/data/gold/features")
csv_path = GOLD_DIR / "gold_drought_features.csv"

df = pd.read_parquet(GOLD_DIR / "gold_drought_features.parquet")

# 1. Danh sách các cột quan trọng được giữ lại
SELECTED_COLUMNS = [
    # Metadata & Tọa độ
    "valid_time",
    "latitude",
    "longitude",
    # Biến khí hậu & độ ẩm đất
    "temp_mean_celsius",
    "dewpoint_celsius",
    "precip_monthly_mm",
    "pet_monthly_mm",
    "water_balance_D",
    "swvl1",
    "swvl2",
    "swvl3",
    "swvl4",
    # Chỉ số hạn hán gốc
    "SPI_1m",
    "SPI_3m",
    "SPI_6m",
    "SPEI_1m",
    "SPEI_3m",
    "SPEI_6m",
    # Features trễ (Lags)
    "temp_mean_celsius_lag_1",
    "temp_mean_celsius_lag_2",
    "temp_mean_celsius_lag_3",
    "precip_monthly_mm_lag_1",
    "precip_monthly_mm_lag_2",
    "precip_monthly_mm_lag_3",
    "pet_monthly_mm_lag_1",
    "pet_monthly_mm_lag_2",
    "pet_monthly_mm_lag_3",
    "SPI_3m_lag_1",
    "SPI_3m_lag_2",
    "SPI_3m_lag_3",
    "SPEI_3m_lag_1",
    "SPEI_3m_lag_2",
    "SPEI_3m_lag_3",
]

# 2. Rút gọn DataFrame
gold_ml_df = df[SELECTED_COLUMNS].copy()

# 3. Ghi đè lại file Parquet & CSV sạch
gold_ml_df.to_parquet(GOLD_DIR / "gold_drought_features.parquet", index=False)
gold_ml_df.to_csv(GOLD_DIR / "gold_drought_features.csv", index=False)

print(
    f"Đã rút gọn xong! Số cột từ 46 giảm xuống còn: {gold_ml_df.shape[1]} cột chuẩn."
)
print("Các cột hiện tại sẵn sàng cho MLP, SVR, Random Forest, XGBoost, LSTM:")
print(list(gold_ml_df.columns))

Đã rút gọn xong! Số cột từ 46 giảm xuống còn: 33 cột chuẩn.
Các cột hiện tại sẵn sàng cho MLP, SVR, Random Forest, XGBoost, LSTM:
['valid_time', 'latitude', 'longitude', 'temp_mean_celsius', 'dewpoint_celsius', 'precip_monthly_mm', 'pet_monthly_mm', 'water_balance_D', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'SPI_1m', 'SPI_3m', 'SPI_6m', 'SPEI_1m', 'SPEI_3m', 'SPEI_6m', 'temp_mean_celsius_lag_1', 'temp_mean_celsius_lag_2', 'temp_mean_celsius_lag_3', 'precip_monthly_mm_lag_1', 'precip_monthly_mm_lag_2', 'precip_monthly_mm_lag_3', 'pet_monthly_mm_lag_1', 'pet_monthly_mm_lag_2', 'pet_monthly_mm_lag_3', 'SPI_3m_lag_1', 'SPI_3m_lag_2', 'SPI_3m_lag_3', 'SPEI_3m_lag_1', 'SPEI_3m_lag_2', 'SPEI_3m_lag_3']


In [8]:
# ==============================================================================
# OPTIMIZED FEATURE SELECTION (CORE 14 FEATURES FOR ML/DL)
# ==============================================================================
from pathlib import Path
import pandas as pd

GOLD_DIR = Path("/home/trang/Drought_prediction/data/gold/features")
df = pd.read_parquet(GOLD_DIR / "gold_drought_features.parquet")

# Danh sách 14 cột cốt lõi
CORE_COLUMNS = [
    # Spatial-Temporal
    "valid_time",
    "latitude",
    "longitude",
    # Current Meteorology & Hydrology
    "temp_mean_celsius",
    "precip_monthly_mm",
    "water_balance_D",
    "swvl1",  # Surface soil moisture
    "swvl3",  # Root-zone soil moisture
    # Key Drought Indices
    "SPI_3m",
    "SPEI_3m",  # Target variable (y) hoặc key feature
    # Core Lags (t-1, t-2)
    "precip_monthly_mm_lag_1",
    "precip_monthly_mm_lag_2",
    "SPEI_3m_lag_1",
    "SPEI_3m_lag_2",
]

# Tinh lọc DataFrame
gold_core_df = df[CORE_COLUMNS].copy()

# Ghi đè file Gold Mart
gold_core_df.to_parquet(GOLD_DIR / "gold_drought_features.parquet", index=False)
gold_core_df.to_csv(GOLD_DIR / "gold_drought_features.csv", index=False)

print("==================================================")
print(" ĐÃ TINH LỌC THÀNH CÔNG BỘ CORE FEATURES TẦNG GOLD")
print("==================================================")
print(f"► Số lượng cột từ 46/33 giảm xuống còn: {gold_core_df.shape[1]} cột")
print(f"► Số bản ghi chuẩn: {len(gold_core_df):,} dòng")

 ĐÃ TINH LỌC THÀNH CÔNG BỘ CORE FEATURES TẦNG GOLD
► Số lượng cột từ 46/33 giảm xuống còn: 14 cột
► Số bản ghi chuẩn: 100,082 dòng


CHUYỂN FILE GOLD PARQUET SANG CSV VÀ HIỂN THỊ 10 DÒNG DỮ LIỆU MẪU

In [9]:

# 1. Đường dẫn file
GOLD_DIR = Path("/home/trang/Drought_prediction/data/gold/features")
parquet_path = GOLD_DIR / "gold_drought_features.parquet"
csv_path = GOLD_DIR / "gold_drought_features.csv"

# 2. Đọc file Parquet và chuyển đổi sang CSV
print("► Đang đọc file Gold Parquet và xuất sang CSV...")
gold_df = pd.read_parquet(parquet_path)
gold_df.to_csv(csv_path, index=False)

print(
    f"Đã chuyển đổi thành công! File CSV được lưu tại: {csv_path}"
)
print(f"• Tổng số dòng: {len(gold_df):,} bản ghi")
print(f"• Tổng số cột: {gold_df.shape[1]} cột")

# 3. Hiển thị 10 dòng dữ liệu đầu tiên
print("\n=== 10 DÒNG DỮ LIỆU ĐẦU TIÊN TẦNG GOLD ===")
display(gold_df.head(10))

► Đang đọc file Gold Parquet và xuất sang CSV...
Đã chuyển đổi thành công! File CSV được lưu tại: /home/trang/Drought_prediction/data/gold/features/gold_drought_features.csv
• Tổng số dòng: 100,082 bản ghi
• Tổng số cột: 14 cột

=== 10 DÒNG DỮ LIỆU ĐẦU TIÊN TẦNG GOLD ===


,valid_time,latitude,longitude,temp_mean_celsius,precip_monthly_mm,water_balance_D,swvl1,swvl3,SPI_3m,SPEI_3m,precip_monthly_mm_lag_1,precip_monthly_mm_lag_2,SPEI_3m_lag_1,SPEI_3m_lag_2
0,2000-06-01,8.6,104.8,27.023956,405.521965,223.868909,0.301346,0.304657,0.816445,0.762089,249.647833,141.970725,-0.406900,-0.914036
1,2000-07-01,8.6,104.8,27.195587,212.653884,1.288602,0.263580,0.252380,1.037827,0.955350,405.521965,249.647833,0.762089,-0.406900
2,2000-08-01,8.6,104.8,26.924713,297.492192,79.011011,0.282471,0.235382,1.187677,1.086303,212.653884,405.521965,0.955350,0.762089
3,2000-09-01,8.6,104.8,26.908600,172.775917,-17.243385,0.263947,0.269669,0.458708,0.411079,297.492192,212.653884,1.086303,0.955350
4,2000-10-01,8.6,104.8,26.391754,441.946718,278.321021,0.316071,0.310211,1.176862,1.186896,172.775917,297.492192,0.411079,1.086303
5,2000-11-01,8.6,104.8,26.355011,227.480164,51.889243,0.276581,0.252029,0.957581,1.110942,441.946718,172.775917,1.186896,0.411079
6,2000-12-01,8.6,104.8,26.139313,159.533033,-37.356091,0.258423,0.270340,0.916104,1.054618,227.480164,441.946718,1.110942,1.186896
7,2001-01-01,8.6,104.8,25.617096,72.086701,-129.342567,0.187286,0.206284,-0.242312,-0.087025,159.533033,227.480164,1.054618,1.110942
8,2001-02-01,8.6,104.8,25.929596,26.102079,-168.624714,0.088745,0.162415,-0.873036,-0.704565,72.086701,159.533033,-0.087025,1.054618
9,2001-03-01,8.6,104.8,26.722321,141.843368,-78.698506,0.181915,0.135620,-0.928441,-0.820342,26.102079,72.086701,-0.704565,-0.087025


In [11]:
# ==============================================================================
# BỔ SUNG YEAR/MONTH VÀ RỜI RẠC HÓA TÊN CỘT TẦNG GOLD (FIXED PATH)
# ==============================================================================
from pathlib import Path
import pandas as pd

# Sử dụng đường dẫn tuyệt đối chuẩn xác
GOLD_DIR = Path("/home/trang/Drought_prediction/data/gold/features")
df = pd.read_parquet(GOLD_DIR / "gold_drought_features.parquet")

# 1. Đảm bảo valid_time là datetime và tách year, month
df["valid_time"] = pd.to_datetime(df["valid_time"])
df["year"] = df["valid_time"].dt.year
df["month"] = df["valid_time"].dt.month

# 2. Map lại tên cột cho tường minh
RENAME_MAP = {
    "water_balance_D": "water_balance_mm",
    "swvl1": "soil_moisture_surface",
    "swvl3": "soil_moisture_rootzone",
    "SPI_3m": "spi_3m",
    "SPEI_3m": "spei_3m",
    "precip_monthly_mm_lag_1": "precip_lag_1m",
    "precip_monthly_mm_lag_2": "precip_lag_2m",
    "SPEI_3m_lag_1": "spei_3m_lag_1m",
    "SPEI_3m_lag_2": "spei_3m_lag_2m",
}

df_clean = df.rename(columns=RENAME_MAP)

# 3. Sắp xếp lại thứ tự cột chuẩn
FINAL_COLUMNS = [
    "valid_time",
    "year",
    "month",
    "latitude",
    "longitude",
    "temp_mean_celsius",
    "precip_monthly_mm",
    "water_balance_mm",
    "soil_moisture_surface",
    "soil_moisture_rootzone",
    "spi_3m",
    "spei_3m",  # Target chính
    "precip_lag_1m",
    "precip_lag_2m",
    "spei_3m_lag_1m",
    "spei_3m_lag_2m",
]

gold_final_clean = df_clean[FINAL_COLUMNS].copy()

# 4. Ghi đè file sản phẩm Gold Mart
gold_final_clean.to_parquet(
    GOLD_DIR / "gold_drought_features.parquet", index=False
)
gold_final_clean.to_csv(GOLD_DIR / "gold_drought_features.csv", index=False)

print("==================================================")
print(" 🎉 ĐÃ ĐỔI TÊN & TÁCH CỘT THỜI GIAN CHUẨN ĐẸP 100%")
print("==================================================")
print(f"► Danh sách 16 cột hoàn chỉnh bàn giao:\n{list(gold_final_clean.columns)}")
display(gold_final_clean.head(3))

 🎉 ĐÃ ĐỔI TÊN & TÁCH CỘT THỜI GIAN CHUẨN ĐẸP 100%
► Danh sách 16 cột hoàn chỉnh bàn giao:
['valid_time', 'year', 'month', 'latitude', 'longitude', 'temp_mean_celsius', 'precip_monthly_mm', 'water_balance_mm', 'soil_moisture_surface', 'soil_moisture_rootzone', 'spi_3m', 'spei_3m', 'precip_lag_1m', 'precip_lag_2m', 'spei_3m_lag_1m', 'spei_3m_lag_2m']


,valid_time,year,month,latitude,longitude,temp_mean_celsius,precip_monthly_mm,water_balance_mm,soil_moisture_surface,soil_moisture_rootzone,spi_3m,spei_3m,precip_lag_1m,precip_lag_2m,spei_3m_lag_1m,spei_3m_lag_2m
0,2000-06-01,2000,6,8.6,104.8,27.023956,405.521965,223.868909,0.301346,0.304657,0.816445,0.762089,249.647833,141.970725,-0.406900,-0.914036
1,2000-07-01,2000,7,8.6,104.8,27.195587,212.653884,1.288602,0.263580,0.252380,1.037827,0.955350,405.521965,249.647833,0.762089,-0.406900
2,2000-08-01,2000,8,8.6,104.8,26.924713,297.492192,79.011011,0.282471,0.235382,1.187677,1.086303,212.653884,405.521965,0.955350,0.762089
